# 04 - Hyperparameter & Optimizer Tuning

**Owner:** Illaria  |  Applied to the **winning architecture** only

Tune learning rate, weight decay, effective batch size, dropout, and compare
optimizers (Adam / AdamW / SGD). See `Project Pipeline.md` -> section 5.

## How to use
1. Run the **Colab bootstrap** cell.
2. Implement `src/tuning.py`.
3. Search over the space and record the best config to reuse in evaluation.

> **GPU equivalent:** `for-gpu/run_tuning.py --variant <winner>` (single node) or `for-gpu/tune_ray.py` (distributed across both GPUs) — Stage 4. Tuned weights -> `model_checkpoints/{variant}_tuning/` and `{variant}_tuned/`.

In [ ]:
# ============================================================
# Colab bootstrap - run this cell first.
# ============================================================
import sys
from pathlib import Path

# Mount Google Drive when running on Colab.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False

# Locate the "codes" folder (the one containing src/ and notebooks/).
if IN_COLAB:
    # TODO: set this to where you uploaded the "codes" folder on Drive.
    CODES_DIR = Path('/content/drive/MyDrive/AI-for-MIA/codes')
else:
    # Local run: this notebook lives in codes/notebooks/, so go up one level.
    CODES_DIR = Path.cwd().parent

assert (CODES_DIR / 'src').exists(), (
    f"Could not find src/ under {CODES_DIR}. "
    "Edit CODES_DIR above to point to your 'codes' folder."
)

# Make `import src...` work.
if str(CODES_DIR) not in sys.path:
    sys.path.insert(0, str(CODES_DIR))

# Auto-reload edited src/*.py without restarting the kernel.
%load_ext autoreload
%autoreload 2

# Shared config resolves the data folder (sibling of "codes").
from src import config
print('CODES_DIR:', config.CODES_DIR)
print('DATA_DIR :', config.DATA_DIR)
assert config.DATA_DIR.exists(), (
    f"Data folder not found at {config.DATA_DIR}. Put the MRNet 'data' folder "
    "next to 'codes', or set os.environ['MRNET_DATA_DIR'] before importing config."
)
print('Setup OK - data folder found.')


In [ ]:
import torch

from src.data_pipeline import build_dataloaders, set_seed
from src.model_factory import build_model
from src import tuning

set_seed(config.SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Tune the winning architecture (e.g. DenseNet121 + CBAM). model_class must be a
# zero-arg callable returning a FRESH model so each trial starts independently.
WINNER_BACKBONE = "densenet121"
USE_CBAM = True

train_loader, val_loader = build_dataloaders(
    task=config.DEFAULT_TASK, plane=config.DEFAULT_PLANE,
    train_augment=config.ORCHESTRATOR_DEFAULTS["train_augment"],  # 'strong'
    batch_size=1, num_workers=0)

def model_class():
    return build_model(backbone=WINNER_BACKBONE, use_cbam=USE_CBAM, dropout=0.0)

# Random search over tuning.SEARCH_SPACE (lr, weight_decay, dropout,
# accumulation_steps, optimizer). Streams trials to a CSV; returns the best.
best_config = tuning.random_search(
    model_class=model_class,
    train_loader=train_loader, val_loader=val_loader, device=device,
    n_trials=10, seed=config.SEED,
    checkpoint_dir=str(config.CHECKPOINTS_DIR / f"{WINNER_BACKBONE}_tuning"),
    results_csv=str(config.GPU_RESULTS_DIR / f"{WINNER_BACKBONE}_tuning_results.csv"),
    task_name=WINNER_BACKBONE,
    epochs=20, patience=5)

print("Best hyperparameters:", best_config)

# For the full distributed/headless sweep used in the report, see the GPU
# equivalents noted at the top of this notebook (run_tuning.py / tune_ray.py).
